# 03 — Aspect-Based Sentiment Analysis

Extract food-domain aspects (taste, price, packaging, delivery, quality, smell) from reviews and classify sentiment per aspect.


In [1]:
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.join('..', 'src'))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from absa import AspectBasedSentimentAnalyzer, SpacyABSAAnalyzer, ASPECT_LIST
from utils import get_logger

logger = get_logger('03_absa')
print('Setup complete ✓')

Setup complete ✓


## 1. Load Data

In [2]:
df = pd.read_parquet(os.path.join('..', 'data', 'reviews_processed.parquet'))

# Sample for speed
SAMPLE = 30_000
df_sample = df.sample(SAMPLE, random_state=42).reset_index(drop=True)
print(f'Loaded {len(df):,} rows, using {SAMPLE:,} sample')

Loaded 568,454 rows, using 30,000 sample


## 2. Run Keyword + VADER ABSA

In [3]:
analyzer = AspectBasedSentimentAnalyzer(context_window=5)
absa_df = analyzer.analyze_batch(df_sample, text_col='Text', id_col='Id')
print(f'Found {len(absa_df):,} aspect mentions across {absa_df["review_id"].nunique():,} reviews')
absa_df.head(10)

2026-05-04 16:24:25 | INFO     | absa | Running ABSA on 30000 reviews …


2026-05-04 16:24:34 | INFO     | absa | ABSA complete. Found 33881 aspect mentions.


Found 33,881 aspect mentions across 21,356 reviews


,review_id,aspect,sentiment,confidence,context_snippet
0,165257,taste,positive,0.2500,makes them a bit too sweet but for me that just
1,427828,taste,neutral,1.0000,tends to have a muddy taste not what i expecte...
2,433955,taste,positive,0.6124,unnecessary personally i like original flavor ...
3,70261,packaging,neutral,1.0000,name on such a small box the ad men must have
4,70261,taste,positive,0.6369,i love the product the taste was refreshing an...
5,49867,taste,positive,0.3182,please add more pineapple flavor to your packa...
6,18984,delivery,neutral,1.0000,to insure continued freshness speedy shipping too
7,138969,taste,neutral,1.0000,don t really have any flavor they just take on...
8,36353,price,neutral,1.0000,come across even the more expensive brands fro...
9,472685,quality,positive,0.4767,am not demeaning senseo s quality but i am cer...


## 3. Aspect Sentiment Summary

In [4]:
summary = analyzer.get_aspect_summary(absa_df)
summary

sentiment,negative,neutral,positive,total
aspect,,,,
taste,1507,2608,9703,13818
price,621,1766,3770,6157
quality,377,1209,3652,5238
packaging,642,2094,1465,4201
delivery,324,1067,1533,2924
smell,281,405,857,1543


## 4. Aspect–Sentiment Heatmap

In [5]:
heat_data = (absa_df.groupby(['aspect', 'sentiment'])
                     .size()
                     .unstack(fill_value=0)
                     .reindex(columns=['negative','neutral','positive'], fill_value=0))

fig = px.imshow(
    heat_data.values,
    x=['Negative', 'Neutral', 'Positive'],
    y=heat_data.index.tolist(),
    color_continuous_scale='RdYlGn',
    text_auto=True,
    title='Aspect × Sentiment Heatmap',
    template='plotly_dark',
)
fig.update_layout(height=400, font_family='DejaVu Sans')
fig.show()

## 5. Most Common Aspects per Sentiment

In [6]:
fig = px.histogram(
    absa_df, x='aspect', color='sentiment',
    barmode='group',
    color_discrete_map={'positive':'#2196f3','negative':'#e63946','neutral':'#f4a261'},
    title='Aspect Frequency by Sentiment',
    template='plotly_dark',
)
fig.update_layout(xaxis_title='Aspect', yaxis_title='Count', font_family='DejaVu Sans')
fig.show()

## 6. Confidence Distribution by Aspect

In [7]:
fig = px.box(
    absa_df, x='aspect', y='confidence', color='sentiment',
    color_discrete_map={'positive':'#2196f3','negative':'#e63946','neutral':'#f4a261'},
    title='Sentiment Confidence by Aspect',
    template='plotly_dark',
)
fig.update_layout(font_family='DejaVu Sans')
fig.show()

## 7. spaCy Dependency Parsing (optional)

In [8]:
# Uncomment to run the spaCy-based variant (slower but more linguistically precise):
# spacy_analyzer = SpacyABSAAnalyzer(model='en_core_web_sm')
# spacy_results = []
# for _, row in df_sample.head(5000).iterrows():
#     spacy_results.extend(spacy_analyzer.analyze_review(row['Id'], row['Text']))
# spacy_df = pd.DataFrame(spacy_results)
# print(f'spaCy ABSA found {len(spacy_df)} aspect mentions')
# spacy_df.head()

## 8. Save Results

In [9]:
OUT = os.path.join('..', 'data', 'absa_results.parquet')
absa_df.to_parquet(OUT, index=False)
print(f'ABSA results saved → {OUT}')

ABSA results saved → ..\data\absa_results.parquet


## Summary

* **Taste** and **quality** are the most frequently mentioned aspects.
* **Delivery** and **packaging** skew negative when mentioned.
* The keyword+VADER approach provides fast, interpretable results.

**Next**: `04_fake_review_detection.ipynb`
